In [56]:
#Packages
import pandas as pd
import numpy as np
import importlib  # Importerer biblioteket importlib, som giver mulighed for at genindlæse moduler
from statsmodels.tsa.ar_model import AutoReg # Importerer AutoReg-klassen fra statsmodels.tsa.ar_model, som bruges til autoregressive modeller

# Import self-defined functions
import sys                              # provides access to Pythons search path
import os                               # Gives Python access to file system functions, such as getting the current working directory
sys.path.append(os.path.abspath(".."))  # Tells python to also seach for modules in the parent directory (where the "functions" folder is located)
from functions import bubble_simulation
from functions import sadf

# Import self-defined functions

importlib.reload(bubble_simulation)  # Genindlæser bubble_simulation.py fra disk og opdaterer modulet i hukommelsen
importlib.reload(sadf)  # Genindlæser SADF_test.py fra disk og opdaterer modulet i hukommelsen


<module 'functions.sadf' from '/Users/tobiasfriumjordhoi/Documents/KU/Bachelor/6. semester/Bachelor/Local repo/functions/sadf.py'>

### Data setup

Vi laver følgende dateserier:

1. Serie med bobbel komponent  
2. Serie uden bobbel komponent  
3. Serie uden bobbel og stød komponent: For at tjekke AR analyse giver koefficient på 1.
4. Vi importerer også P_t.csv som dataserie

In [57]:
# 1.
Pf, B, P = bubble_simulation.simulate_bubble() # Kører funktionen simulate_bubble() fra bubble_simulation-modulet og gemmer resultaterne i variablerne Pf, B og P

# 2.
Pf_nobubble, B_nobubble, P_nobubble = bubble_simulation.simulate_bubble(R=0, B0=0, sigma_b=0) # Simulate bubble med intet bobble komponent. 

# 3. Create dataset with no shock variabel and bubbel component
Pf_noshock, B_noshock, P_noshock = bubble_simulation.simulate_bubble(R=0, B0=0, sigma_b=0, sigma_f=0) 

# 4. Load CSV file
csv_file = pd.read_csv("../data/P_t.csv") # ".." means go up one folder
P_t_csv = np.asarray(csv_file["Price"])  # Converts the column "P_t" in the csv-file to a numpy-array

# Comparing the dataset witb and without bubble component.
print("Price with bubble: \n", P[98:103])  # Prints first 5 elements of the numpy array
print("Price without bubble: \n", P_nobubble[98:103])  # Prints first 5 elements of the numpy array

# Comparing the dataset with no shock and no bubble component.
print("Price with no shock and no bubble: \n", P_noshock[98:103])  # Prints first 5 elements of the numpy array

#
print("P_t.csv series: \n", P_t_csv[98:103])  # Prints first 5 elements of the numpy array



Price with bubble: 
 98     15.536247
99     16.186404
100    16.226555
101     5.389514
102     4.894246
Name: Price, dtype: float64
Price without bubble: 
 98     5.449905
99     4.995212
100    4.294286
101    5.389514
102    4.894246
Name: Price, dtype: float64
Price with no shock and no bubble: 
 98     10.0
99     10.0
100    10.0
101    10.0
102    10.0
Name: Price, dtype: float64
P_t.csv series: 
 [ 7.90924304  8.29820237  7.79253811  0.50464082 -0.54622732]


### AR analyse
Analyse af tidsseriene beskrevet ovenfor:$

3. For at tjekke gyldigheden af de simulerede data kan vi estimere en AR(1)-model på prisserien uden chok- og boblekomponent. Vi forventer en koefficient på 1, da $\rho=1$, hvilket giver $P_t = P_{t-1}$ 

In [58]:
# 3. Estimate AR(1) model on the dataset with no shock and no bubble component
model = AutoReg(P_noshock, lags=1, trend='n')   # AR(1) model uden konstantled (trend='n') og med 1 lag (lags=1)
results = model.fit()        # Estimate parameters

print(results.summary())     # Print regression table

                            AutoReg Model Results                             
Dep. Variable:                  Price   No. Observations:                  200
Model:                     AutoReg(1)   Log Likelihood                5901.325
Method:               Conditional MLE   S.D. of innovations              0.000
Date:                Fri, 27 Feb 2026   AIC                         -11798.651
Time:                        16:40:08   BIC                         -11792.064
Sample:                             1   HQIC                        -11795.985
                                  200                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Price.L1       1.0000   2.27e-16   4.41e+15      0.000       1.000       1.000
                                    Roots                                    
                  Real          Imaginary           M

### SADF test

WE now perform the SADF test for the following dataseries:
1. P_t.csv

In [59]:
#1. 
sadf_stat, adf_path, r_path = sadf.SADF_test(
    P_t_csv,
    r0=0.05,
    lags=0,
    trend='c'
)

print("SADF statistic:", sadf_stat)

SADF statistic: -1.9801006521670779


### Kritiske værdier